<a href="https://colab.research.google.com/github/JWKang81/MedScan/blob/main/Gresearch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import pandas as pd
import cudf
import gresearch_crypto
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split

#from sklearn.preprocessing import StandardScaler
###########################
#訓練模型
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset, Dataset

##matplot
import matplotlib.pyplot as plt
from matplotlib.pyplot import figure


train_path = '../input/g-research-crypto-forecasting/train.csv'
asset_details_path = '../input/g-research-crypto-forecasting/asset_details.csv'

ModuleNotFoundError: No module named 'gresearch_crypto'

In [ ]:
device = 'cpu'
if torch.cuda.is_available():
    device = 'cuda'
print(f'Using {device} for training.')

In [ ]:
###dataset
df_train = cudf.read_csv(train_path)
df_train = df_train.to_pandas()
df_train.replace([np.inf, -np.inf], np.nan, inplace=True) #去除inf(以nan取代), VMAP中有 +- inf
df_train = df_train.dropna(how = 'any') #any:刪除有nan column
df_train['Upper_shadow'] = df_train['High'] - (df_train[['Open','Close']]).max(axis=1)
df_train['Lower_shadow'] = (df_train[['Open','Close']]).min(axis=1) - df_train['Low']
df_train.head()

In [ ]:
asset_details = pd.read_csv(asset_details_path, dtype=
                           {'Asset_ID': 'int8', 'Weight': 'float64', 'Asset_Name': 'str'})
names = {}
for idx, row in asset_details.iterrows():
    names[row.iloc[0]] = row.iloc[2]

In [ ]:
df_train = df_train[df_train['timestamp']<1623542400] #不動用OVERLAP資料, 6/13以後
df_train.shape

In [ ]:
### model parameter
epochs = 30
TEST_SIZE = 0.2

LEARING_RATE = 0.01 #learning rate
NUM_WORKERS = 2
BATCH_SIZE = 30

FEATURES = ['Close','High','Low','Open','VWAP','Volume','Lower_shadow','Upper_shadow']
FEATURES_NUM = len(FEATURES)
TARGET = 'Target'
SEQ_LENGTH = 60 #拿多少時段當time series 拿半天當訓練

In [ ]:
##定義貨幣DATASET
class AssetDataset(Dataset):
    def __init__(self, data, features, target, seq_len = SEQ_LENGTH):
        '''
        data: 資料集
        features: 訓練、測試資料所有特徵名稱
        target: 資料目標label
        seq_len: 時間序列長度
        '''
        self.data = data
        self.data_len = len(data)

        self.features = features
        self.target = target
        self.seq_len = seq_len

        self.ftseries = self.feature_target_series() #綁

    def feature_target_series(self):
        results = []
        for i in range(self.data_len - self.seq_len): #將資料劃分成一段段時間組成
            x = self.data[i:i + self.seq_len][self.features].values
            y = self.data[i + self.seq_len:i + self.seq_len + 1][self.target].values  ##記得target是下一段時間的資料, 我是拿前30分鐘的資料預測下一分鐘  這個一分鐘可依需求改變
            results.append((x,y))
        return results

    def __len__(self):
        return len(self.ftseries)

    def __getitem__(self, idx):
        return self.ftseries[idx]

In [ ]:
class LSTM(nn.Module):
    def __init__(self, input_size, hidden_layer, layers, output_size, direction = 1): #layers幾層 directions: 1為單向, 2雙向
        super(LSTM, self).__init__()

        self.hidden_layer = hidden_layer
        self.layers = layers
        self.direction = direction

        self.lstm = nn.LSTM(input_size, hidden_layer, layers, batch_first = True, dropout = 0.2) #記得加batch_first 他才以time series  (batch, seq, feature)

        self.dropout = nn.Dropout(0.2)

        self.fc1 = nn.Linear(hidden_layer, output_size)
        #self.float()


    def init_hidden_weight(self, batchsize): #1st: 單向或雙向 2nd: 放batch, 3rd 放隱藏
        hidden_layer_state = (self.layers*self.direction, batchsize, self.hidden_layer)
        return (torch.zeros(hidden_layer_state).to(device), torch.zeros(hidden_layer_state).to(device)) #h_0 跟 c_0

    def forward(self, x, state):
        #lstm
        out, (hn, cn) = self.lstm(x, state)  #hn,cn: hidden state最終狀態
        out = self.fc1(out)

        return out, (hn, cn) #final

In [ ]:
def run_epoch(model, loader, criterion, optimizer, is_training=False):
    epoch_loss = 0
    if is_training:
        model.train()
    else:
        model.eval()

    state = model.init_hidden_weight(BATCH_SIZE)

    for idx, (x, target) in enumerate(loader):

        x = x.to(device)

        target = target.to(device)

       # target = target.reshape(-1)

        state = [s.detach() for s in state] #不進行反向傳播

        if is_training:
            optimizer.zero_grad()

        out, state = model(x.float(), state)

        loss = criterion(out[:, -1, :].contiguous(), target.contiguous().float())

        if is_training:
            loss.backward()
            optimizer.step()

        epoch_loss += (loss.detach().item() / BATCH_SIZE)

    lr = scheduler.get_last_lr()[0]

    return epoch_loss, lr

In [ ]:
def show_assetInfo(train, test):
    data_len = len(train) + len(test)

    train_target = train['Target'].values
    train_num = len(train_target)
    train_date = train['timestamp'].astype('datetime64[s]').dt.strftime('%Y-%m-%d %H:%M').values

    train_name = names[train_data['Asset_ID'].iloc[0]]

    print(f'Train data from {train_date[0]} to {train_date[-1]}')


    fig = figure(figsize=(25, 5), dpi=60)
    fig.patch.set_facecolor((1.0, 1.0, 1.0))

    plt.plot(train_date, train_target, color="#001f3f")

    #定義x軸級距
    xticks = [train_date[i] if ((i%720==0 and (train_num-i) > 720) or i==train_num-1) else None for i in range(train_num)] # make x ticks nice
    x = np.arange(0,len(xticks))
    plt.xticks(x, xticks, rotation='vertical')


    plt.title(train_name)

    plt.grid(b=None, which='major', axis='y', linestyle='--')
    plt.show()

In [ ]:
models = {} #儲存各式MODEL

for i in range(14):

    #準備train, test data
    train_data, test_data = train_test_split(df_train[df_train['Asset_ID'] == i][-14400:], test_size=TEST_SIZE, shuffle=False) #8成當訓練, 用10天訓練
    print(f'Size of {names[i]} train dataset: {train_data.shape}, test dataset: {test_data.shape}')
    #show_assetInfo(train_data, test_data)

    #準備dataset
    train_dataset = AssetDataset(train_data, FEATURES, TARGET)
    test_dataset = AssetDataset(test_data, FEATURES, TARGET)

    #準備dataloader
    train_loader = DataLoader(train_dataset, batch_size = BATCH_SIZE, num_workers = NUM_WORKERS , shuffle = True)
    test_loader = DataLoader(test_dataset, batch_size = BATCH_SIZE, num_workers = NUM_WORKERS, shuffle = False)
    print(f'{names[i]}: There are {len(train_loader)} training batches and {len(test_loader)} testing batches')


    # 準備model參數
    mymodel = None
    mymodel = LSTM(FEATURES_NUM, BATCH_SIZE, 2, 1) #feature, batch,

    if device == 'cuda':
        mymodel.cuda()

    criterion = nn.MSELoss()

    optimizer = torch.optim.SGD(mymodel.parameters(), lr = LEARING_RATE, momentum=0.9)

    scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=40, gamma=0.1)

    #data = data.reindex(range(data.index[0], data.index[-1]+60,60), method='pad')#更改index為時間戳

    for epoch in range(epochs):
        loss_train, lr_train = run_epoch(mymodel, train_loader, criterion, optimizer, True)
        loss_test, lr_test = run_epoch(mymodel, test_loader, criterion, optimizer)
        scheduler.step()

        print('Epoch[{}/{}] | loss train:{:.6f}, test:{:.6f} | lr:{:.6f}'
              .format(epoch+1, epochs, loss_train, loss_test, lr_train))

    models[i] = mymodel

In [ ]:
import gresearch_crypto
env = gresearch_crypto.make_env()
iter_test = env.iter_test()
for (test_df, sample_prediction_df) in iter_test:
    test_df['Upper_shadow'] = test_df['High'] - (test_df[['Open','Close']]).max(axis=1)
    test_df['Lower_shadow'] = (test_df[['Open','Close']]).min(axis=1) - test_df['Low']
    for _, row in test_df.iterrows():

        model = models[row['Asset_ID']]

        if device == 'cuda':
            model.cuda()
        model.eval()

        test = np.array([row[FEATURES].values])

        test = torch.tensor(test)

        test = test.view(1, -1, FEATURES_NUM)

        #記得初始h,c
        state = model.init_hidden_weight(1)
        state = [s.detach() for s in state]

        out, (hn,cn) = model(test.float().to(device), state)

        pridict = out[:,-1, :].item()

        sample_prediction_df.loc[sample_prediction_df['row_id'] == row['row_id'], 'Target'] = pridict

    #print(sample_prediction_df)
    env.predict(sample_prediction_df)

In [ ]:
gresearch_crypto.make_env.__called__ = False #寫完記得註解掉